# Уравнение теплопроводности: численная реализация на Python

## Теоретическая часть

Рассмотрим моделирование распространения тепла в идеальном стержне. Предположим, что у нас есть цилиндрический стержень, концы которого поддерживаются при фиксированной температуре, и он нагревается в определенной точке x в течение определенного интервала времени.

Математическая формулировка задачи:

$$\frac{\partial u}{\partial t} = k \frac{\partial^2 u}{\partial x^2}$$

где:
- $u(x,t)$ - температура в точке x в момент времени t
- $k$ - коэффициент температуропроводности
- $0 < x < L$, $t > 0$

Граничные условия:
- $u(0,t) = u(L,t) = T_0$ (температура на концах фиксирована)
- $u(x,0) = f(x)$ (начальное распределение температуры)

Для частного случая $f(x) = \sin(x)$ и $L = \pi$ решение имеет вид:

$$u(x,t) = T_0 + e^{-kt} \sin(x)$$

## Практическая реализация

Импортируем необходимые библиотеки:

In [ ]:
import numpy as np
from numpy import pi
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Для корректного отображения в Google Colab
%matplotlib inline

### Настройка параметров

Задаем основные параметры модели:

In [ ]:
# Коэффициент температуропроводности
k_diffusion = 2

# Масштабный коэффициент (для визуализации)
scale = 5

# Длина стержня (0,L) по оси x
L = pi

# Начальные условия: u(0,t) = u(L,t) = 0
# Температура в точках x=0 и x=L фиксирована
x0 = np.linspace(0, L, 1000)

# Начальное время
t0 = 0

# Температура стержня в состоянии покоя (до нагрева)
temp0 = 5

# Шаг по времени
dt = 0.01

# Количество временных шагов
num_steps = 200

### Определение функций

Аналитическое решение уравнения теплопроводности:

$$u(x,t) = T_0 + e^{-kt} \sin(x)$$

In [ ]:
def u(x, t):
    """
    Функция температуры u(x,t) = T0 + e^(-kt)*sin(x)
    
    Параметры:
    x - координата по длине стержня
    t - время
    
    Возвращает:
    Температура в точке x в момент времени t
    """
    return temp0 + scale * np.exp(-k_diffusion * t) * np.sin(x)

Градиент функции температуры:

$$\nabla u(x,t) = \left(\frac{\partial u}{\partial x}, \frac{\partial u}{\partial t}\right) = \left(e^{-kt} \cos(x), -k e^{-kt} \sin(x)\right)$$

In [ ]:
def grad_u(x, t):
    """
    Градиент функции температуры: (du/dx, du/dt)
    
    Параметры:
    x - координата по длине стержня
    t - время
    
    Возвращает:
    Вектор градиента [du/dx, du/dt]
    """
    return scale * np.array([np.exp(-k_diffusion * t) * np.cos(x), 
                           -k_diffusion * np.exp(-k_diffusion * t) * np.sin(x)])

### Численное моделирование

Используем приближение для численного решения:

$$u(x, t + \Delta t) \approx u(x,t) + \frac{\partial u}{\partial t} \Delta t$$

In [ ]:
# Массивы для хранения результатов
temperature_data = []  # Значения температуры
time_values = []      # Моменты времени

# Численное интегрирование
current_time = t0
for i in range(num_steps):
    # Применяем численное приближение
    value = u(x0, current_time) + grad_u(x0, current_time)[1] * dt
    
    # Сохраняем текущее время
    time_values.append(current_time)
    
    # Увеличиваем время
    current_time = current_time + dt
    
    # Сохраняем значение температуры
    temperature_data.append(value)

print(f"Вычислено {len(temperature_data)} временных шагов")

### Визуализация результатов

Создаем анимацию для визуализации распространения тепла:

In [ ]:
# Создаем фигуру для анимации
fig, ax = plt.subplots(figsize=(10, 6))
fig.set_dpi(100)

# Настраиваем график
ax.set_xlim(0, L)
ax.set_ylim(temp0-2, 2.5*scale)
ax.grid(True)
ax.set_xlabel('Положение x')
ax.set_ylabel('Температура')
ax.set_title('Уравнение теплопроводности')

# Создаем первоначальный график
line, = ax.plot(x0, temperature_data[0], color='red', linewidth=2)
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)

def animate(frame):
    """
    Функция анимации: показывает эволюцию температуры со временем
    в каждой точке x стержня
    """
    if frame < len(temperature_data):
        # Обновляем данные графика
        line.set_ydata(temperature_data[frame])
        time_text.set_text(f'Время: {time_values[frame]:.2f} с')
    return line, time_text

# Создаем анимацию
anim = animation.FuncAnimation(fig, animate, frames=len(temperature_data), 
                              interval=50, blit=True)

# Для отображения в Google Colab
plt.close()  # Предотвращаем дублирование отображения
HTML(anim.to_jshtml())

## Анализ результатов

Из анимации видно:

1. **Начальное состояние**: При t=0 температура распределена по синусоидальному закону
2. **Процесс остывания**: Со временем температура выравнивается
3. **Стационарное состояние**: В пределе t→∞ температура стремится к $T_0$

Скорость остывания определяется коэффициентом $k$:

$$\tau = \frac{1}{k}$$

где $\tau$ - характерное время остывания.